# S1.3 · Heavy tails and the laws of bibliometrics

*Phase S, statistics and probability · S1, part 3 of 4 · about 3 hours*

Where do skewed distributions come from, and how do you describe their tails? This lesson simulates cumulative advantage, fits power laws properly, and tests them against the alternative that often fits as well, the lognormal. It ends with the three classic laws of bibliometrics: Lotka's law for authors, Bradford's for journals and Zipf's for words, the last one tested on Dante's *Commedia*.

**Run this lesson:** [in your browser (JupyterLite)](https://sbrzt.github.io/personal-course/lite/lab/index.html?path=lessons/phase-s/s1-3-heavy-tails-laws.ipynb) · [in Google Colab](https://colab.research.google.com/github/sbrzt/personal-course/blob/main/lessons/phase-s/s1-3-heavy-tails-laws.ipynb)

**By the end you can**

- simulate Price's model, and explain how cumulative advantage produces inequality;
- draw samples from a power law by inverse-transform sampling;
- fit a power-law exponent by maximum likelihood, and explain why a straight line on a log–log plot is not evidence;
- compare a power law with a lognormal using a likelihood-ratio test;
- state Lotka's, Bradford's and Zipf's laws, and check them on data.

In [ ]:
# ⚙️ Setup: run this cell first
import sys, pathlib, subprocess

if sys.platform == "emscripten":            # in the browser (JupyterLite)
    import piplite
    await piplite.install(["personal-course"])
elif "google.colab" in sys.modules:         # in Google Colab
    repo = "git+https://github.com/sbrzt/personal-course"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"personal-course @ {repo}"], check=True)
else:                                       # on your computer
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:    # find the course folder
        if (folder / "course" / "__init__.py").exists():
            sys.path.insert(0, str(folder))
            break

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import powerlaw
from course import data, models, plots, stats
from course.checks import s1_3 as check

warnings.filterwarnings("ignore", module="powerlaw")   # its optimiser is chatty
plots.use_style()
print("Ready.")

## 1 · Where heavy tails come from

In 1968 the sociologist Robert K. Merton named the **Matthew effect** after a line in the Gospel of Matthew: "unto every one that hath shall be given". Well-known scientists receive more credit for similar work, which makes them better known still. In 1976 Derek de Solla Price turned the idea into a model of citation, **cumulative advantage**:

- papers arrive one at a time;
- each new paper cites *m* earlier papers;
- the chance that an earlier paper *i* is cited is proportional to $c_i + a$, where $c_i$ is the number of citations it already has, and $a > 0$ gives uncited papers a chance too.

### ✏️ Exercise 1 · Simulating cumulative advantage

Complete the line that computes the weights of the earlier papers.

In [ ]:
def price_model(n_papers=2000, m=3, a=1.0, seed=0):
    rng = np.random.default_rng(seed)
    cites = np.zeros(n_papers, dtype=int)
    for t in range(m, n_papers):     # paper t cites m of papers 0, ..., t-1
        weights = ...                # one weight per earlier paper
        p = weights / weights.sum()
        cited = rng.choice(t, size=m, replace=False, p=p)
        cites[cited] += 1
    return cites

check.ex1(price_model)

<details>
<summary>💡 Hint</summary>

`cites[:t]` holds the citation counts of papers 0 to t − 1 so far.

</details>

<details>
<summary>✅ Solution</summary>

```python
def price_model(n_papers=2000, m=3, a=1.0, seed=0):
    rng = np.random.default_rng(seed)
    cites = np.zeros(n_papers, dtype=int)
    for t in range(m, n_papers):
        weights = cites[:t] + a
        p = weights / weights.sum()
        cited = rng.choice(t, size=m, replace=False, p=p)
        cites[cited] += 1
    return cites

check.ex1(price_model)
```

With `weights = cites[:t]` alone, every paper starts with weight 0, every probability is 0 divided by 0, and the simulation fails. The constant *a* is a paper's "initial attractiveness": the larger it is compared with *m*, the more citing is driven by chance rather than by past success.

</details>

From here on we use `models.price`, a faster version of the same model. Compare it with a world where every earlier paper is equally likely to be cited:

In [ ]:
c_adv = models.price(n_papers=5000, m=3, a=1.0, seed=1)
c_uni = models.uniform_attachment(n_papers=5000, m=3, seed=1)

fig, ax = plt.subplots()
plots.ccdf(c_adv, ax=ax, label="cumulative advantage (a = 1)", color=plots.BLUE)
plots.ccdf(c_uni, ax=ax, label="every earlier paper equally likely",
           color=plots.ORANGE)
ax.set(xlabel="citations", ylabel="share of papers ≥ x",
       title="Same number of citations, different tails")
ax.legend()
plt.show()

### ✏️ Exercise 2 · Luck versus advantage

For each value of *a* in 0.1, 1, 10 and 1000, simulate `models.price(n_papers=3000, m=3, a=a, seed=1)` and compute the Gini coefficient of the citations with `stats.gini`. Store the results in the dictionary `gini_by_a`, with the values of *a* as keys.

In [ ]:
gini_by_a = {}
for a in [0.1, 1, 10, 1000]:
    gini_by_a[a] = ...

check.ex2(gini_by_a)

<details>
<summary>💡 Hint</summary>

Inside the loop: simulate with the current `a`, then pass the result to `stats.gini`.

</details>

<details>
<summary>✅ Solution</summary>

```python
gini_by_a = {}
for a in [0.1, 1, 10, 1000]:
    gini_by_a[a] = stats.gini(models.price(n_papers=3000, m=3, a=a, seed=1))

check.ex2(gini_by_a)
```

With *a* = 0.1, nearly all citations go to a few early papers (Gini 0.99). With *a* = 1000, citing is almost random, yet inequality does not disappear (Gini 0.57): older papers have simply had more chances to be cited. Two lessons for metascience. Unequal citations do not show, by themselves, that papers differ in quality. And age must be taken into account whenever papers of different ages are compared.

</details>

> **🤖 Why this matters for AI.** Cumulative advantage is a feedback loop, and AI tools can tighten it. Search engines that rank papers partly by citations, and language models that suggest the references they saw most often during training, both make already-cited work more visible, which in the model above means a smaller *a*. In S1.2 you met evidence that LLMs favour highly cited papers more than human authors do.

## 2 · What a power law is

A quantity follows a **power law** above some threshold $x_{\min}$ if its probability density is

$$p(x) = \frac{\alpha - 1}{x_{\min}} \left(\frac{x}{x_{\min}}\right)^{-\alpha}, \qquad x \ge x_{\min},$$

with exponent $\alpha > 1$. Its CCDF is $P(X \ge x) = (x / x_{\min})^{-(\alpha - 1)}$: on log–log axes, a straight line with slope $-(\alpha - 1)$. Two consequences:

- the distribution is *scale-free*: doubling *x* always divides the density by the same factor, $2^{\alpha}$, whatever *x* is;
- if $\alpha \le 3$ the variance is infinite, and if $\alpha \le 2$ even the mean is: sample means and SDs then never settle, however much data you collect.

**Inverse-transform sampling** turns uniform random numbers into draws from any distribution whose CDF you can invert: if *U* is uniform on (0, 1) and *F* is a CDF, then $F^{-1}(U)$ has CDF *F*. For the power law, $F(x) = 1 - (x / x_{\min})^{-(\alpha - 1)}$, so

$$F^{-1}(u) = x_{\min} \, (1 - u)^{-1 / (\alpha - 1)}.$$

### ✏️ Exercise 3 · Sampling a power law

Write `sample_power_law(n, alpha, xmin, rng)` with the formula above.

In [ ]:
def sample_power_law(n, alpha, xmin, rng):
    u = rng.random(n)                          # n uniform numbers in [0, 1)
    return ...

check.ex3(sample_power_law)

<details>
<summary>💡 Hint</summary>

In numpy, `a ** b` raises `a` to the power `b`, element by element.

</details>

<details>
<summary>✅ Solution</summary>

```python
def sample_power_law(n, alpha, xmin, rng):
    u = rng.random(n)
    return xmin * (1 - u) ** (-1 / (alpha - 1))

check.ex3(sample_power_law)
```

</details>

In [ ]:
x = stats.sample_power_law(10_000, alpha=2.5, xmin=1.0, rng=0)
fig, ax = plt.subplots()
plots.ccdf(x, ax=ax, label="10,000 draws", color=plots.BLUE)
grid = np.logspace(0, np.log10(x.max()), 50)
ax.plot(grid, grid ** -1.5, color=plots.INK, linestyle="--", linewidth=1.2,
        label="theory: slope −(α − 1) = −1.5")
ax.set(xlabel="x", ylabel="share ≥ x",
       title="A power law is a straight line on a log–log CCDF")
ax.legend()
plt.show()

### Fitting the exponent

The tempting method is to draw the data on log–log axes, fit a straight line by least squares and read off its slope. Clauset, Shalizi and Newman (2009) showed that this gives biased and unreliable estimates. Use **maximum likelihood** instead: choose the α under which the observed data are most probable. For a continuous power law the answer has a closed form,

$$\hat\alpha = 1 + n \left[ \sum_{i=1}^{n} \ln \frac{x_i}{x_{\min}} \right]^{-1}, \qquad \text{with standard error } \frac{\hat\alpha - 1}{\sqrt{n}},$$

where the sum runs over the *n* observations with $x_i \ge x_{\min}$. You will derive it in S3, where maximum likelihood also turns out to be the way language models are trained.

### ✏️ Exercise 4 · The maximum-likelihood exponent

In [ ]:
def alpha_mle(x, xmin):
    x = np.asarray(x, dtype=float)
    x = x[x >= xmin]                           # the formula uses the tail only
    return ...

check.ex4(alpha_mle)

<details>
<summary>💡 Hint</summary>

`np.log` is the natural logarithm, and `x.size` is *n*.

</details>

<details>
<summary>✅ Solution</summary>

```python
def alpha_mle(x, xmin):
    x = np.asarray(x, dtype=float)
    x = x[x >= xmin]
    return 1 + x.size / np.sum(np.log(x / xmin))

check.ex4(alpha_mle)
```

</details>

How much better is maximum likelihood? Draw 300 samples of 500 values with α = 2.5, and estimate α from each sample twice: by maximum likelihood, and by least squares on a log–log histogram.

In [ ]:
rng = np.random.default_rng(42)
mle, lsq = [], []
for _ in range(300):
    x = stats.sample_power_law(500, 2.5, 1.0, rng)
    mle.append(stats.alpha_mle(x, 1.0))
    edges = np.logspace(0, np.log10(x.max()) + 1e-9, 20)   # log-spaced bins
    density, _ = np.histogram(x, bins=edges, density=True)
    middle = np.sqrt(edges[1:] * edges[:-1])
    keep = density > 0                     # empty bins have no logarithm
    lsq.append(-np.polyfit(np.log(middle[keep]), np.log(density[keep]), 1)[0])

fig, ax = plt.subplots()
bins = np.linspace(1.8, 3.0, 40)
ax.hist(lsq, bins=bins, histtype="step", linewidth=2, color=plots.ORANGE,
        label="least squares")
ax.hist(mle, bins=bins, histtype="step", linewidth=2, color=plots.BLUE,
        label="maximum likelihood")
ax.axvline(2.5, color=plots.INK, linestyle="--", linewidth=1)
ax.set(xlabel="estimated α (true value 2.5)", ylabel="number of samples",
       title="Same data, two estimators", ylim=(0, 75))
ax.legend(loc="upper left")
plt.show()
print(f"maximum likelihood: mean {np.mean(mle):.2f}, SD {np.std(mle):.2f}")
print(f"least squares:      mean {np.mean(lsq):.2f}, SD {np.std(lsq):.2f}")

Least squares is biased (its estimates centre near 2.27, not 2.5) and more than twice as variable. Its errors come from the choices it forces on you, such as the bins and what to do with empty ones, and from treating the noisy tail bins as if they were as reliable as the full ones.

## 3 · Power law or lognormal?

A variable is **lognormal** if its logarithm is normally distributed. On a log–log CCDF a lognormal bends downwards gently, and over two or three orders of magnitude it can look almost straight. Many heavy-tailed data that were once called power laws turn out to fit a lognormal as well or better. For complete citation data, including the many papers with few citations, a pure power law fits poorly: Thelwall (2016) found that a discretised lognormal or a "hooked" power law describes such data best. Broido and Clauset (2019) tested nearly a thousand networks and found that strongly "scale-free" ones are rare.

The `powerlaw` package (Alstott et al. 2014) does the careful work:

- `powerlaw.Fit(x, discrete=True)` chooses $x_{\min}$ by making the fitted power law as close as possible to the data above it (the Kolmogorov–Smirnov distance), then fits α by maximum likelihood. Use `discrete=True` for counts.
- `fit.distribution_compare("power_law", "lognormal", normalized_ratio=True)` returns *R* and *p*. *R* is a normalised log-likelihood ratio: positive if the first distribution fits better, negative if the second does. *p* is the probability of a ratio at least this far from zero if the two fitted equally well. A large *p* (say above 0.1) means the data cannot tell the two apart.

### ✏️ Exercise 5 · Power law or lognormal?

Fit the 5,000 simulated citation counts from S1.1 and compare the power law with the lognormal. Report `alpha_hat`, `xmin_hat`, `R` and `p`.

In [ ]:
cites = data.citation_counts()
fit = ...
alpha_hat = ...
xmin_hat = ...
R = ...
p = ...

check.ex5(alpha_hat, xmin_hat, R, p)

<details>
<summary>💡 Hint</summary>

After `fit = powerlaw.Fit(cites, discrete=True, verbose=False)`, the fitted values are `fit.power_law.alpha` and `fit.power_law.xmin`. `distribution_compare` returns a pair: `R, p = fit.distribution_compare(...)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
cites = data.citation_counts()
fit = powerlaw.Fit(cites, discrete=True, verbose=False)
alpha_hat = fit.power_law.alpha
xmin_hat = fit.power_law.xmin
R, p = fit.distribution_compare("power_law", "lognormal", normalized_ratio=True)

check.ex5(alpha_hat, xmin_hat, R, p)
```

</details>

In [ ]:
from matplotlib.ticker import NullFormatter

fit = powerlaw.Fit(cites, discrete=True, verbose=False)
fig, ax = plt.subplots()
fit.plot_ccdf(ax=ax, color=plots.INK, linewidth=2, label="data (tail above xmin)")
fit.power_law.plot_ccdf(ax=ax, color=plots.BLUE, linestyle="--",
                        label=f"power law, α = {fit.power_law.alpha:.2f}")
fit.lognormal.plot_ccdf(ax=ax, color=plots.ORANGE, linestyle="--", label="lognormal")
ax.set(xlabel="citations", ylabel="share ≥ x", title="Two models, one tail")
ax.xaxis.set_minor_formatter(NullFormatter())      # fewer crowded tick labels
ax.legend()
plt.show()
n_tail = int((cites >= fit.power_law.xmin).sum())
print(f"Tail used for the fit: {n_tail} papers ({n_tail / cites.size:.0%} of all)")
print(f"alpha = {fit.power_law.alpha:.2f} ± {fit.power_law.sigma:.2f}")

Three things to notice.

1. The fit describes the **tail only**: the 411 papers with at least 26 citations, 8% of the field. α says nothing about the other 92%.
2. *R* is negative (the lognormal fits slightly better), but *p* = 0.32: the data cannot tell the two models apart.
3. For an infinitely large field, Price's model predicts a power-law tail with α = 2 + *a* / *m*, here 2 + 8/10 = 2.8. The estimate, 2.47 ± 0.07, is lower: with 5,000 papers the tail has not reached its limit. A fitted exponent is a description of finite data, not a reading of the mechanism.

A fair summary: "the tail is heavy, with α around 2.5 above 26 citations; a power law and a lognormal describe it equally well."

> **🤖 Why this matters for AI.** "Neural scaling laws" state that a language model's loss falls as a power law of its size, its training data and its compute (Kaplan et al. 2020; Hoffmann et al. 2022). They are fitted to a few dozen or hundred training runs, often as straight lines on log–log axes, and they guide how money is spent on training. When Besiroglu et al. (2024) tried to replicate one of Hoffmann et al.'s estimates, they found it inconsistent with the paper's other two methods, and its confidence intervals so narrow that they would require over 600,000 experiments, where the authors probably ran fewer than 500. The questions you asked here (which range was fitted, by which method, how uncertain, against which alternatives) are the ones to ask of such claims.

## 4 · The three laws of bibliometrics

### Lotka's law (1926): authors

Alfred Lotka counted how many chemists and physicists had published 1, 2, 3, … papers. The number of authors with *k* papers is roughly proportional to $1 / k^{n}$ with *n* ≈ 2. With *n* = 2, the share of authors with a single paper is $1/\zeta(2) = 6/\pi^2 \approx 61\%$.

`data.authorship()` lists the authors of 1,600 simulated articles, one row per pair of article and author.

In [ ]:
authorship = data.authorship()
authorship.head()

### ✏️ Exercise 6 · Lotka's law

Compute `table`, a Series whose index is the number of articles *k* (1, 2, 3, …) and whose values are the number of authors with *k* articles, sorted by *k*. Then compute `share_single`, the share of authors with exactly one article.

In [ ]:
per_author = ...              # number of articles of each author
table = ...
share_single = ...

check.ex6(table, share_single)

<details>
<summary>💡 Hint</summary>

`groupby("author").size()` counts the rows of each author. `value_counts()` then counts how many authors have each count, and `sort_index()` orders the result by *k*.

</details>

<details>
<summary>✅ Solution</summary>

```python
per_author = authorship.drop_duplicates().groupby("author").size()
table = per_author.value_counts().sort_index()
share_single = table.loc[1] / table.sum()

check.ex6(table, share_single)
```

</details>

The share of single-article authors also gives you an estimate of the exponent: under Lotka's law it equals $1/\zeta(n)$, which falls as *n* grows, so you can solve for *n*.

In [ ]:
from scipy.optimize import brentq

table = stats.productivity(authorship)
share = table.loc[1] / table.sum()
n_hat = brentq(lambda n: stats.lotka_share_single(n) - share, 1.2, 6)
print(f"one-article authors: {share:.3f}  ->  Lotka exponent n ≈ {n_hat:.2f}")

fig, ax = plt.subplots()
ax.loglog(table.index, table.values, "o", color=plots.BLUE,
          label="authors with k articles")
k = np.arange(1, table.index.max() + 1)
ax.loglog(k, table.loc[1] * k ** -2.0, color=plots.INK, linestyle="--",
          linewidth=1.2, label="Lotka, n = 2")
ax.set(xlabel="k (articles)", ylabel="number of authors", title="Lotka's law")
ax.legend()
plt.show()

### Bradford's law (1934): journals

Samuel Bradford, a librarian at the Science Museum in London, ranked journals by how many articles each had published on a subject, then cut the ranked list into zones holding about the same number of articles. The numbers of journals in the zones grew roughly as 1 : *k* : *k*²: a small core of journals publishes a third of the literature, and the last third is scattered over very many journals. This "scattering" is why exhaustive literature searches are so expensive.

### ✏️ Exercise 7 · Bradford's zones

Write `my_bradford_zones(counts, n_zones=3)`:

1. sort the counts from largest to smallest;
2. for each journal, compute `before`, the number of articles in all the journals ranked above it;
3. put each journal in zone `floor(n_zones * before / total)`;
4. return the number of journals in each zone.

In [ ]:
def my_bradford_zones(counts, n_zones=3):
    c = np.sort(np.asarray(counts))[::-1]   # most productive journal first
    before = ...           # articles in the journals ranked above each one
    zone = ...             # 0, 1, ..., n_zones - 1
    return np.bincount(zone, minlength=n_zones)

check.ex7(my_bradford_zones)

<details>
<summary>💡 Hint</summary>

`np.cumsum(c) - c` gives, for each journal, the sum of the counts before it. `np.floor(...).astype(int)` turns the zone numbers into integers.

</details>

<details>
<summary>✅ Solution</summary>

```python
def my_bradford_zones(counts, n_zones=3):
    c = np.sort(np.asarray(counts))[::-1]
    before = np.cumsum(c) - c
    zone = np.floor(n_zones * before / c.sum()).astype(int)
    return np.bincount(zone, minlength=n_zones)

check.ex7(my_bradford_zones)
```

There are other conventions, such as assigning a journal to the zone where most of its articles fall. With many journals they give similar results.

</details>

In [ ]:
journals = data.journal_counts()
zones = stats.bradford_zones(journals)
print(f"{journals.sum():,} articles in {journals.size} journals")
print("journals per zone:", zones.tolist())
print(f"zone 2 / zone 1 = {zones[1] / zones[0]:.1f}")
print(f"zone 3 / zone 2 = {zones[2] / zones[1]:.1f}")

Three journals publish a third of the articles; 223 share the last third. The two ratios (8.0 and 9.3) are close, as Bradford's law says they should be, with *k* ≈ 8.6.

### Zipf's law (1949): words

George Kingsley Zipf ranked the words of texts by frequency. The frequency of the word of rank *r* is roughly proportional to $1 / r^{s}$, with *s* close to 1: the most frequent word occurs about twice as often as the second, three times as often as the third, and so on. A rank–frequency law with exponent *s* corresponds to a power law in the *frequency distribution* (how many words occur *f* times) with exponent $\alpha = 1 + 1/s$, which is how we will estimate *s*.

Our text is Dante's *Commedia*: 100 cantos, in the public domain.

In [ ]:
text = data.commedia()
print(text[:300])

### ✏️ Exercise 8 · Counting Dante's words

Lower-case the text and split it into words with the pattern `WORD` (runs of letters, accented ones included; apostrophes split words, so *l'amor* gives *l* and *amor*). Then compute:

- `tokens`, the list of all words in order;
- `freqs`, an array with the frequency of each distinct word, most frequent first;
- `hapax_share`, the share of distinct words that occur exactly once (the *hapax legomena*).

In [ ]:
from collections import Counter
WORD = stats.WORD                              # re.compile(r"[^\W\d_]+")

tokens = ...
counts = ...                                   # a Counter: word -> frequency
freqs = ...
hapax_share = ...

check.ex8(tokens, freqs, hapax_share)

<details>
<summary>💡 Hint</summary>

`WORD.findall(text.lower())` returns the list of words. `Counter(tokens)` counts them; `counts.values()` are the frequencies.

</details>

<details>
<summary>✅ Solution</summary>

```python
from collections import Counter
WORD = stats.WORD

tokens = WORD.findall(text.lower())
counts = Counter(tokens)
freqs = np.array(sorted(counts.values(), reverse=True))
hapax_share = np.mean(freqs == 1)

check.ex8(tokens, freqs, hapax_share)
```

</details>

In [ ]:
counts = Counter(stats.tokenize(text))
freqs = stats.rank_frequency(stats.tokenize(text))
print("most frequent:", counts.most_common(10))

fig, ax = plt.subplots()
plots.rank_frequency(freqs, ax=ax, label="Commedia", color=plots.BLUE)
r = np.arange(1, freqs.size + 1)
ax.loglog(r, freqs[0] / r, color=plots.INK, linestyle="--", linewidth=1.2,
          label="Zipf, s = 1")
ax.set(title="Zipf's law in the Commedia")
ax.legend()
plt.show()

The most frequent words are function words (*e*, *che*, *la*, *a*, *di*, *non*), and the elided *l* and *ch* come from forms such as *l'amor* and *ch'i'*.

### ✏️ Exercise 9 · Zipf's exponent

Fit a discrete power law to `freqs` with `powerlaw.Fit`, store its exponent in `alpha_words`, and convert it to Zipf's exponent: `zipf_s = 1 / (alpha_words - 1)`.

In [ ]:
fit_words = ...
alpha_words = ...
zipf_s = ...

check.ex9(alpha_words, zipf_s)

<details>
<summary>✅ Solution</summary>

```python
fit_words = powerlaw.Fit(freqs, discrete=True, verbose=False)
alpha_words = fit_words.power_law.alpha
zipf_s = 1 / (alpha_words - 1)

check.ex9(alpha_words, zipf_s)
```

Zipf's exponent comes out at 1.11, close to the classic 1. Here the fit starts at frequency 1, so it covers every word. But a close fit does not explain much: Piantadosi (2014) shows that many different mechanisms produce Zipf-like curves, so the law tells you little about how language works.

</details>

> **🤖 Why this matters for AI.** Zipf's law explains a basic design choice in language models. A vocabulary of whole words can never be complete: 57% of the distinct words in the *Commedia* occur only once, and every new text brings new ones. So modern tokenisers split rare words into frequent pieces (byte-pair encoding: Sennrich et al. 2016), and a vocabulary of a few tens of thousands of sub-word units can then spell any word. Two consequences follow. Text in languages that are rare in the training data is cut into more pieces, so it costs more and fits less into a model's context window (Petrov et al. 2023). And facts that appear rarely in the training data are learned poorly: the long tail of knowledge (Kandpal et al. 2023). In S2 you will turn word counts like these into a small language model.

## 5 · Wrap-up

### ✍️ Explain it

Your supervisor shows you a log–log plot of the citations received by datasets, with a straight line through it, and says: "dataset citations follow a power law, so a rich-get-richer process must be at work." In at most 150 words, say what you would check before agreeing.

*Your answer:*

<details>
<summary>A model answer</summary>

First, how the line was fitted: least squares on a log–log plot is biased, so I would refit α by maximum likelihood, with a standard error, and state which part of the data (above which xmin) the fit describes. Second, the alternatives: a lognormal or a hooked power law often fits citation data as well, so I would compare them with a likelihood-ratio test and report R and p. Third, how much data the tail holds: with a few dozen datasets above xmin, almost any heavy-tailed model will fit. Finally, the mechanism: a power law is consistent with cumulative advantage but does not prove it, because other processes produce heavy tails too, and differences in age alone create inequality. Testing a mechanism needs data on how citations accumulate over time.

</details>

### Check your progress

In [ ]:
check.summary();

### Self-check

1. Why does a power law with α = 2.2 have no finite variance, and what does that mean for the SD of a sample?
2. What does `powerlaw.Fit` choose for you, and on what criterion?
3. Lotka's, Bradford's and Zipf's laws describe different things. What do they have in common?

<details>
<summary>Answers</summary>

1. The variance is infinite for α ≤ 3. The SD of a sample is always finite, but it keeps growing, erratically, as the sample grows: it never settles on a value, so do not report it as a property of the data.
2. The threshold xmin, chosen to minimise the Kolmogorov–Smirnov distance between the data above xmin and the fitted power law. α is then the maximum-likelihood estimate for that xmin.
3. Each describes a skewed "who gets how much" distribution (papers among authors, articles among journals, occurrences among words), with a few large units and very many small ones, and each can be written as a power law.

</details>

### Done when

- [ ] Exercises 1 to 9 show ✅ in `check.summary()`.
- [ ] You can explain why least squares on a log–log plot is not a good estimator.
- [ ] You have written your 150-word answer.

### Further reading

- Clauset, A., Shalizi, C. R., & Newman, M. E. J. (2009). Power-law distributions in empirical data. *SIAM Review*, 51(4), 661–703. <https://doi.org/10.1137/070710111>
- Newman, M. E. J. (2005). Power laws, Pareto distributions and Zipf's law. *Contemporary Physics*, 46(5), 323–351. <https://doi.org/10.1080/00107510500052444>
- Alstott, J., Bullmore, E., & Plenz, D. (2014). powerlaw: A Python package for analysis of heavy-tailed distributions. *PLoS ONE*, 9(1), e85777. <https://doi.org/10.1371/journal.pone.0085777>
- Price, D. de S. (1976). A general theory of bibliometric and other cumulative advantage processes. *Journal of the American Society for Information Science*, 27(5), 292–306. <https://doi.org/10.1002/asi.4630270505>
- Merton, R. K. (1968). The Matthew effect in science. *Science*, 159(3810), 56–63. <https://doi.org/10.1126/science.159.3810.56>
- Thelwall, M. (2016). The discretised lognormal and hooked power law distributions for complete citation data: Best options for modelling and regression. *Journal of Informetrics*, 10(2), 336–346. <https://doi.org/10.1016/j.joi.2015.12.007>
- Broido, A. D., & Clauset, A. (2019). Scale-free networks are rare. *Nature Communications*, 10, 1017. <https://doi.org/10.1038/s41467-019-08746-5>
- Piantadosi, S. T. (2014). Zipf's word frequency law in natural language: A critical review and future directions. *Psychonomic Bulletin & Review*, 21(5), 1112–1130. <https://doi.org/10.3758/s13423-014-0585-6>
- Kaplan, J., et al. (2020). Scaling laws for neural language models. arXiv:2001.08361. <https://arxiv.org/abs/2001.08361>
- Hoffmann, J., et al. (2022). Training compute-optimal large language models. arXiv:2203.15556. <https://arxiv.org/abs/2203.15556>
- Besiroglu, T., Erdil, E., Barnett, M., & You, J. (2024). Chinchilla scaling: A replication attempt. arXiv:2404.10102. <https://arxiv.org/abs/2404.10102>
- Sennrich, R., Haddow, B., & Birch, A. (2016). Neural machine translation of rare words with subword units. *Proceedings of ACL 2016*, 1715–1725. <https://doi.org/10.18653/v1/P16-1162>
- Petrov, A., La Malfa, E., Torr, P. H. S., & Bibi, A. (2023). Language model tokenizers introduce unfairness between languages. *NeurIPS 2023*. arXiv:2305.15425. <https://arxiv.org/abs/2305.15425>
- Kandpal, N., Deng, H., Roberts, A., Wallace, E., & Raffel, C. (2023). Large language models struggle to learn long-tail knowledge. *ICML 2023*. arXiv:2211.08411. <https://arxiv.org/abs/2211.08411>

**Next:** S1.4 · Project: the shape of Bibliothecae\.it.